# Projekt mp3 loesung

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">MP3-Verwaltung: Von der Idee zur Implementierung</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python &nbsp;|&nbsp; Kapitel 13: Abschlussprojekt &nbsp;|&nbsp; Loesung und Herleitung</p>
</div>
</div>

> **[Kursinhalt]** Dieses Notebook zeigt den vollstaendigen Loesungsweg -- von der Problemanalyse bis zur lauffaehigen Implementierung. Es dient der Nachbereitung und dem Verstaendnis der Entwurfsentscheidungen.

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Das Problem verstehen -- bevor eine Zeile Code entsteht
</span>
</div>

Wer viele MP3-Dateien hat, kennt das Problem: Die Sammlung waechst ueber Jahre. Dateien landen in verschiedenen Ordnern. Dateinamen wie `track01.mp3` oder `Unbekannt - Unbekannt.mp3` sagen nichts. Manche Songs sind doppelt vorhanden, mit leicht unterschiedlichen Dateinamen. Die eingebetteten Metadaten -- Interpret, Titel, Album -- stimmen manchmal nicht mit dem Dateinamen ueberein oder fehlen ganz.

Das Programm soll diesem Chaos Herr werden. Konkret:

- Ein Verzeichnis nach MP3-Dateien durchsuchen (auch Unterordner)
- Die eingebetteten Metadaten auslesen
- Die Sammlung anzeigen, sortierbar und durchsuchbar
- Den Zustand speichern -- damit man nicht bei jedem Start neu scannt

**Bevor man anfaengt zu programmieren, lohnt sich eine Frage: Was ist hier eigentlich schwierig?**

Nicht die GUI. Nicht das Speichern. Die zwei wirklich interessanten Probleme sind:

1. **Metadaten in MP3-Dateien lesen** -- MP3-Dateien koennen Metadaten in verschiedenen Formaten einbetten. Das gaengigste heisst ID3. Wir brauchen eine Bibliothek die das versteht, und muessen damit umgehen dass Tags fehlen oder in unerwarteten Formaten vorliegen.

2. **Fehlende Metadaten sinnvoll behandeln** -- nicht jede MP3-Datei hat einen Titel-Tag. Das Programm darf deshalb nicht abstuerzen. Es muss eine Fallback-Strategie haben.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. Was sind ID3-Tags?
</span>
</div>

Eine MP3-Datei besteht aus zwei Teilen: den Audiodaten (die eigentliche Musik) und einem optionalen Metadatenblock am Anfang oder Ende der Datei. Dieser Block heisst ID3-Header.

Der ID3-Header enthaelt sogenannte Frames -- jeder Frame hat einen vierstelligen Schluessel und einen Wert:

| Schluessel | Bedeutung | Beispielwert |
|---|---|---|
| `TIT2` | Titel | `Bohemian Rhapsody` |
| `TPE1` | Interpret (Artist) | `Queen` |
| `TALB` | Album | `A Night at the Opera` |
| `TDRC` | Aufnahmedatum/Jahr | `1975` oder `1975-11-21` |
| `TRCK` | Track-Nummer | `11` oder `11/17` |
| `APIC` | Albumcover (Bild-Bytes) | _(binaere Daten)_ |

Die Schluessel sind standardisiert (ID3v2.3 und ID3v2.4), aber aeltere Dateien koennen ID3v1 haben -- ein kompakteres Format mit fixen Feldgroessen. Das Paket `mutagen` abstrahiert den Unterschied weitgehend.

**Die wichtigste Erkenntnis:** Jeder dieser Frames kann fehlen. `TIT2` fehlt bei einer Datei die nie korrekt getaggt wurde. `TDRC` kann ein vollstaendiges Datum enthalten (`1975-11-21`) statt nur das Jahr. Das Programm muss mit all diesen Faellen umgehen.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. Erster Kontakt mit mutagen
</span>
</div>

mutagen ist eine Python-Bibliothek zum Lesen und Schreiben von Audiometadaten. Installation:

```bash
pip install mutagen
```

Zwei Klassen sind fuer uns relevant: `MP3` liest die technischen Audioinformationen (Dauer, Bitrate), `ID3` liest die Metadaten-Tags.

In [ ]:
# Der naive erste Ansatz -- direkt auf Tags zugreifen
# Was passiert wenn ein Tag fehlt?

from mutagen.mp3 import MP3
from mutagen.id3 import ID3
from pathlib import Path

# Wir erstellen eine Testdatei um die Beispiele ausfuehren zu koennen
# (normalerweise wuerde man hier eine echte MP3-Datei angeben)
import tempfile, os
from mutagen.id3 import TIT2, TPE1, TALB, TDRC

testdir = Path(tempfile.mkdtemp())

# Datei MIT vollstaendigen Tags
pfad_voll = testdir / 'bohemian.mp3'
pfad_voll.write_bytes(bytes([0xFF, 0xFB, 0x90, 0x00]) + bytes(128))
tags = ID3()
tags['TIT2'] = TIT2(encoding=3, text='Bohemian Rhapsody')
tags['TPE1'] = TPE1(encoding=3, text='Queen')
tags['TALB'] = TALB(encoding=3, text='A Night at the Opera')
tags['TDRC'] = TDRC(encoding=3, text='1975-11-21')  # vollstaendiges Datum!
tags.save(pfad_voll)

# Datei OHNE Tags
pfad_leer = testdir / 'unbekannt.mp3'
pfad_leer.write_bytes(bytes([0xFF, 0xFB, 0x90, 0x00]) + bytes(128))

print('Testdateien erstellt.')
print(f'  Mit Tags:    {pfad_voll.name}')
print(f'  Ohne Tags:   {pfad_leer.name}')

In [ ]:
# --- Naiver Ansatz: direkt auf Tags zugreifen ---
# Wir lesen die Datei MIT Tags -- das funktioniert.

tags = ID3(pfad_voll)

titel    = str(tags['TIT2'])
interpret = str(tags['TPE1'])
album    = str(tags['TALB'])
jahr     = str(tags['TDRC'])

print(f'Titel:    {titel}')
print(f'Interpret: {interpret}')
print(f'Album:    {album}')
print(f'Jahr:     {jahr}')  # Achtung: '1975-11-21' -- nicht nur das Jahr!

# Problem 1: TDRC enthaelt manchmal ein vollstaendiges Datum
# Problem 2: Wenn ein Tag fehlt, wirft tags['TIT2'] einen KeyError
# Problem 3: str(tags['TIT2']) gibt ein Tag-Objekt als String aus, nicht direkt den Text

# Naiven Ansatz auf die Datei OHNE Tags:
from mutagen.id3 import ID3NoHeaderError
try:
    tags_leer = ID3(pfad_leer)
    print(str(tags_leer['TIT2']))  # KeyError!
except ID3NoHeaderError:
    print('Datei ohne Tags: ID3NoHeaderError')
except KeyError as e:
    print(f'Tag fehlt: {e}')

Drei Probleme sichtbar:

1. `ID3NoHeaderError` wenn die Datei gar keine ID3-Tags hat
2. `KeyError` wenn ein bestimmter Tag fehlt
3. `TDRC` kann `'1975-11-21'` enthalten statt nur `'1975'` -- wir wollen nur das Jahr

Der naive Ansatz faellt bei jedem davon um. Wir brauchen eine robustere Strategie.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. Robustes Tag-Lesen: jeder Fehler einzeln abgefangen
</span>
</div>

Das Prinzip: jeden einzelnen Tag-Zugriff in eine eigene Hilfsfunktion kapseln, die `None` zurueckgibt statt abzusturzen. Und die aeussere `ID3NoHeaderError`-Exception separat abfangen.

In [ ]:
# --- Robuster Ansatz: Hilfsfunktionen fangen jeden Fehler einzeln ab ---

def lese_tag(tags, schluessel):
    """
    Liest einen Tag-Wert als String.
    Gibt None zurueck wenn der Tag fehlt oder leer ist.
    """
    try:
        wert = str(tags[schluessel])
        return wert.strip() or None   # leere Strings -> None
    except KeyError:
        return None


def lese_tag_jahr(tags):
    """
    Liest nur das Jahr aus TDRC.
    '1975-11-21' -> '1975'
    '1975'       -> '1975'
    fehlt        -> None
    """
    try:
        wert = str(tags['TDRC']).strip()
        return wert[:4] if wert else None   # erste 4 Zeichen = Jahr
    except KeyError:
        return None


def lese_song_robust(pfad):
    """
    Liest eine MP3-Datei -- robust gegen fehlende Tags.
    Gibt immer ein Dictionary zurueck, nie eine Exception.
    """
    from mutagen.id3 import ID3NoHeaderError

    # Schritt 1: Audiodaten (Dauer) -- separat von den Tags
    try:
        audio = MP3(pfad)
        dauer = audio.info.length
    except Exception:
        dauer = 0.0

    # Schritt 2: ID3-Tags -- komplett optional
    titel = interpret = album = jahr = None
    try:
        tags = ID3(pfad)
        titel    = lese_tag(tags, 'TIT2')
        interpret = lese_tag(tags, 'TPE1')
        album    = lese_tag(tags, 'TALB')
        jahr     = lese_tag_jahr(tags)
    except ID3NoHeaderError:
        pass   # keine Tags -- alle Felder bleiben None

    return {
        'pfad':      str(pfad),
        'titel':     titel,
        'interpret': interpret,
        'album':     album,
        'jahr':      jahr,
        'dauer':     dauer,
    }


# Test: beide Dateien
for pfad in [pfad_voll, pfad_leer]:
    daten = lese_song_robust(pfad)
    print(f"\n{pfad.name}:")
    for k, v in daten.items():
        if k != 'pfad':
            print(f"  {k:10}: {v}")

Die Datei ohne Tags liefert jetzt `None` fuer alle Felder -- kein Absturz. Das ist die Grundregel fuer Daten aus externen Quellen: **lieber `None` als Exception, lieber explizit als implizit.**

Und das Jahr `'1975-11-21'` wird sauber auf `'1975'` reduziert -- `wert[:4]` schneidet genau die ersten vier Zeichen aus, egal wie lang der String danach ist.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Entwurfsentscheidung: Warum @dataclass fuer Song?
</span>
</div>

Ein Song hat fuenf Dateneigenschaften: Pfad, Titel, Interpret, Album, Jahr, Dauer. Man koennte das als Dictionary speichern -- `song['titel']`. Oder als einfache Klasse mit `__init__`.

Wir entscheiden uns fuer `@dataclass`. Warum?

Ein normales Dictionary hat keine Methoden -- `song['titel_anzeige']` geht nicht, man muesste ueberall `song['titel'] or Path(song['pfad']).stem` schreiben. Eine normale Klasse wuerde einen `__init__`, `__repr__` und `__eq__` brauchen den man von Hand schreibt.

`@dataclass` erledigt `__init__`, `__repr__` und `__eq__` automatisch aus den Felddeklarationen heraus. Wir koennen trotzdem Methoden und Properties hinzufuegen.

In [ ]:
# Vergleich: Dictionary vs. normale Klasse vs. @dataclass

from dataclasses import dataclass
from pathlib import Path
from typing import Optional

# --- Option A: Dictionary ---
song_dict = {
    'pfad': '/musik/bohemian.mp3',
    'titel': 'Bohemian Rhapsody',
    'interpret': 'Queen',
}
# Zugriff: song_dict['titel']
# Fallback: song_dict['titel'] or Path(song_dict['pfad']).stem  -- ueberall wiederholen
# Vergleich: song_dict == anderes_dict  -- funktioniert, aber unuebersichtlich
# Methoden: nicht moeglich

# --- Option B: @dataclass ---
@dataclass
class Song:
    pfad:      str
    titel:     Optional[str] = None
    interpret: Optional[str] = None
    album:     Optional[str] = None
    jahr:      Optional[str] = None
    dauer:     float         = 0.0

    # Properties kapseln die Fallback-Logik einmalig
    @property
    def titel_anzeige(self):
        return self.titel or Path(self.pfad).stem

    @property
    def interpret_anzeige(self):
        return self.interpret or 'Unbekannt'

    @property
    def dauer_anzeige(self):
        if not self.dauer:
            return '–'
        return f"{int(self.dauer)//60}:{int(self.dauer)%60:02d}"

    def __str__(self):
        return f"{self.interpret_anzeige} – {self.titel_anzeige} ({self.dauer_anzeige})"


s1 = Song('/musik/bohemian.mp3', 'Bohemian Rhapsody', 'Queen', 'A Night at the Opera', '1975', 354.0)
s2 = Song('/musik/unbekannt.mp3')   # alle Felder optional

print(s1)                        # __str__ automatisch
print(repr(s1))                  # __repr__ automatisch (von @dataclass)
print(s1 == s1)                  # __eq__ automatisch
print(s2.titel_anzeige)          # Fallback auf Dateiname
print(s2.interpret_anzeige)      # Fallback auf 'Unbekannt'
print(s1.dauer_anzeige)          # '5:54'

Das `@property`-Muster ist hier zentral: Die Fallback-Logik (`or Path(...).stem`) steht **einmal** in der Klasse -- nicht ueberall im Programm wo ein Titel angezeigt wird. Aendert sich die Fallback-Logik, muss man sie nur an einer Stelle anpassen.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
6. Persistenz: Song-Objekte als JSON speichern und laden
</span>
</div>

Das Scannen einer grossen Musiksammlung kann Minuten dauern. Die Bibliothek soll deshalb gespeichert werden -- beim naechsten Start wird die JSON-Datei geladen statt alles neu zu scannen.

Das Problem: `json.dump()` versteht keine Python-Objekte, nur Dictionaries, Listen, Strings und Zahlen. Wir brauchen eine Umwandlung in beide Richtungen: `Song -> dict` und `dict -> Song`.

In [ ]:
import json
from dataclasses import asdict

# asdict() ist eine Funktion aus dem dataclasses-Modul.
# Sie wandelt eine @dataclass-Instanz in ein Dictionary um -- kostenlos.

song = Song('/musik/bohemian.mp3', 'Bohemian Rhapsody', 'Queen', 'A Night at the Opera', '1975', 354.0)

# Song -> Dictionary
d = asdict(song)
print('Als Dictionary:')
print(d)
print()

# Dictionary -> JSON-String
json_str = json.dumps(d, ensure_ascii=False, indent=2)
print('Als JSON:')
print(json_str[:200])   # nur Anfang anzeigen

In [ ]:
# Dictionary -> Song: mit Schutz gegen unbekannte Felder
# (wenn das JSON-Format sich in einer zukuenftigen Version aendert,
#  sollen alte Dateien trotzdem geladen werden koennen)

def song_aus_dict(daten):
    bekannte_felder = {'pfad', 'titel', 'interpret', 'album', 'jahr', 'dauer'}
    gefiltert = {k: v for k, v in daten.items() if k in bekannte_felder}
    return Song(**gefiltert)


# Roundtrip-Test: Song -> dict -> JSON -> dict -> Song
original     = Song('/musik/bohemian.mp3', 'Bohemian Rhapsody', 'Queen', 'A Night at the Opera', '1975', 354.0)
als_json_str = json.dumps(asdict(original))
als_dict     = json.loads(als_json_str)
rekonstruiert = song_aus_dict(als_dict)

print(f'Original:      {original}')
print(f'Rekonstruiert: {rekonstruiert}')
print(f'Identisch:     {original == rekonstruiert}')   # True dank @dataclass __eq__

# Test: unbekanntes Feld im JSON wird ignoriert
mit_extra_feld = {'pfad': '/a.mp3', 'titel': 'Test', 'unbekanntes_feld': 'ignoriert'}
song_aus_dict(mit_extra_feld)   # kein Fehler
print('Unbekannte Felder werden ignoriert: OK')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
7. Die Bibliothek-Klasse: Suche, Sortierung und Duplikate
</span>
</div>

Eine Liste von Songs allein reicht nicht. Wir brauchen Operationen darauf: suchen, sortieren, Duplikate finden. Diese Operationen gehoeren in eine eigene Klasse -- die `Bibliothek`.

Warum eine Klasse statt einfach Funktionen? Eine Funktion `suchen(songs, begriff)` wuerde die Liste als Parameter brauchen. Ueberall im Programm wuerde `songs` weitergegeben. Eine Klasse kapselt die Liste intern und stellt Methoden bereit -- der Rest des Programms muss die interne Struktur nicht kennen.

In [ ]:
# Suche -- case-insensitiv, Teilstrings, mehrere Felder gleichzeitig

# Testdaten
songs = [
    Song('/a.mp3', 'Bohemian Rhapsody',       'Queen',        'A Night at the Opera', '1975', 354.0),
    Song('/b.mp3', 'Stairway to Heaven',      'Led Zeppelin', 'Led Zeppelin IV',      '1971', 482.0),
    Song('/c.mp3', 'Hotel California',        'Eagles',       'Hotel California',     '1977', 391.0),
    Song('/d.mp3', 'Smells Like Teen Spirit', 'Nirvana',      'Nevermind',            '1991', 301.0),
    Song('/e.mp3',  None,                     'The Beatles',  'Abbey Road',           '1969', 0.0),
]


def suchen(songs, suchbegriff):
    """
    Gibt alle Songs zurueck die den Suchbegriff in Titel,
    Interpret oder Album enthalten.
    Case-insensitive, Teilstrings.
    """
    if not suchbegriff.strip():
        return list(songs)   # leere Suche -> alles zurueck

    begrif = suchbegriff.lower()
    ergebnis = []
    for song in songs:
        # In mehreren Feldern gleichzeitig suchen
        felder = [
            song.titel_anzeige,
            song.interpret_anzeige,
            song.album_anzeige,
        ]
        if any(begrif in f.lower() for f in felder):
            ergebnis.append(song)
    return ergebnis


# Tests
print('Suche "queen":')
for s in suchen(songs, 'queen'):
    print(f'  {s}')

print('\nSuche "HEAVEN" (Grossbuchstaben):')
for s in suchen(songs, 'HEAVEN'):
    print(f'  {s}')

print('\nSuche "1969" (nur im Album-Jahr):')
# Jahr ist kein Suchfeld -- 'Abbey Road' wird nicht gefunden wenn man '1969' sucht
# Das ist eine bewusste Entscheidung -- man sucht nach Namen, nicht nach Metadaten
for s in suchen(songs, '1969'):
    print(f'  {s}')
print('  (leer -- Jahr ist kein Suchfeld)')

In [ ]:
# Sortierung -- mit None-Behandlung
# Das Problem: ein Song ohne Interpret hat interpret=None.
# sorted() kann None nicht mit Strings vergleichen.

# Falscher Ansatz:
try:
    sorted(songs, key=lambda s: s.interpret)   # TypeError: '<' not supported between None and str
except TypeError as e:
    print(f'Fehler beim naiven Sortieren: {e}')

print()

# Richtiger Ansatz: None-Werte ans Ende schieben
def sort_key(song, feld, absteigend=False):
    wert = getattr(song, feld)   # dynamisch das gewuenschte Feld lesen
    if wert is None:
        # None immer ans Ende -- unabhaengig von der Sortierrichtung
        return 'zzz' if not absteigend else ''
    return str(wert).lower()   # case-insensitive


sortiert = sorted(songs, key=lambda s: sort_key(s, 'interpret'))
print('Sortiert nach Interpret (None ans Ende):')
for s in sortiert:
    print(f'  {s.interpret_anzeige:15} | {s.titel_anzeige}')

In [ ]:
# Duplikaterkennung
# Zwei Songs gelten als Duplikat wenn Interpret UND Titel uebereinstimmen.
# Strategie: Gruppierung per Dictionary -- Schluessel ist 'interpret|titel'

songs_mit_duplikat = songs + [
    Song('/kopie.mp3', 'Bohemian Rhapsody', 'Queen', 'Greatest Hits', '1981', 354.0),   # Duplikat!
]

def finde_duplikate(songs):
    gruppen = {}
    for song in songs:
        if not song.titel or not song.interpret:
            continue   # Songs ohne Metadaten koennen nicht verglichen werden
        # Schluessel: normalisiert auf Kleinschreibung
        schluessel = f"{song.interpret.lower()}|{song.titel.lower()}"
        gruppen.setdefault(schluessel, []).append(song)

    # Nur Gruppen mit mehr als einem Song zurueckgeben
    return [gruppe for gruppe in gruppen.values() if len(gruppe) > 1]


duplikate = finde_duplikate(songs_mit_duplikat)
print(f'{len(duplikate)} Duplikatgruppe(n) gefunden:')
for gruppe in duplikate:
    print(f'  {gruppe[0].interpret_anzeige} - {gruppe[0].titel_anzeige}:')
    for s in gruppe:
        print(f'    {s.pfad}')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
8. Verzeichnis rekursiv scannen mit pathlib
</span>
</div>

`pathlib.Path.rglob()` durchsucht ein Verzeichnis rekursiv -- also auch alle Unterordner. Das `r` in `rglob` steht fuer *recursive*.

In [ ]:
import tempfile
from pathlib import Path

# Testverzeichnis mit Unterordnern anlegen
basis = Path(tempfile.mkdtemp()) / 'musik'
(basis / 'Rock').mkdir(parents=True)
(basis / 'Pop' / 'Alben').mkdir(parents=True)

# Einige Dummy-Dateien anlegen
dateien = [
    basis / 'song01.mp3',
    basis / 'Rock' / 'song02.mp3',
    basis / 'Rock' / 'song03.MP3',       # Grossbuchstaben!
    basis / 'Pop' / 'Alben' / 'song04.mp3',
    basis / 'Pop' / 'notiz.txt',          # keine MP3
]
for d in dateien:
    d.write_bytes(b'')

# rglob findet rekursiv alle .mp3-Dateien
gefunden_klein = list(basis.rglob('*.mp3'))
gefunden_gross = list(basis.rglob('*.MP3'))

print('*.mp3 (Kleinbuchstaben):', [f.name for f in gefunden_klein])
print('*.MP3 (Grossbuchstaben):', [f.name for f in gefunden_gross])

# Beide zusammenfassen und Duplikate entfernen
# dict.fromkeys() entfernt Duplikate und behaelt die Reihenfolge
alle_mp3 = list(dict.fromkeys(sorted(gefunden_klein + gefunden_gross)))
print(f'\nInsgesamt: {len(alle_mp3)} MP3-Dateien')
for p in alle_mp3:
    print(f'  {p.relative_to(basis)}')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
9. Fehlertoleranz beim Scan: eine schlechte Datei stoppt nicht alles
</span>
</div>

Was passiert wenn eine Datei korrupt ist, kein echtes MP3 ist, oder die Rechte fehlen? Ein `raise` wuerde den gesamten Scan abbrechen -- alle danach folgenden Dateien werden nicht eingelesen.

Die bessere Strategie: Fehler pro Datei abfangen, in eine Fehlerliste schreiben, und weitermachen. Am Ende wird die Fehlerliste zusammen mit den Songs zurueckgegeben.

In [ ]:
# Das Muster: Fehler sammeln statt abbrechen

def scanne_verzeichnis(verzeichnis, fortschritt_callback=None):
    """
    Gibt immer (songs, fehler) zurueck -- nie eine Exception.
    Fehler werden gesammelt, der Scan laeuft weiter.
    """
    verzeichnis = Path(verzeichnis)

    # Verzeichnis existiert? Hier darf eine Exception geworfen werden --
    # ein fehlendes Verzeichnis ist ein Programmierfehler, kein Datenfehler.
    if not verzeichnis.exists():
        raise FileNotFoundError(f'Nicht gefunden: {verzeichnis}')

    mp3_dateien = sorted(
        list(dict.fromkeys(
            list(verzeichnis.rglob('*.mp3')) +
            list(verzeichnis.rglob('*.MP3'))
        ))
    )

    songs  = []
    fehler = []   # Fehlermeldungen als Strings

    for i, pfad in enumerate(mp3_dateien, start=1):
        # Fortschritt melden wenn Callback angegeben
        if fortschritt_callback:
            fortschritt_callback(i, len(mp3_dateien), pfad.name)

        try:
            song = lese_song_robust(pfad)
            songs.append(Song(**song))
        except Exception as e:
            # Diese Datei ueberspringen, aber nicht den gesamten Scan stoppen
            fehler.append(f'{pfad.name}: {e}')

    return songs, fehler


# Demonstration mit Fortschritts-Callback
def zeige_fortschritt(aktuell, gesamt, dateiname):
    print(f'  [{aktuell}/{gesamt}] {dateiname}')

songs_gefunden, fehler_liste = scanne_verzeichnis(basis, zeige_fortschritt)
print(f'\n{len(songs_gefunden)} Songs eingelesen, {len(fehler_liste)} Fehler.')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
10. Modularer Aufbau: Warum drei Dateien?
</span>
</div>

Das Projekt besteht aus drei Dateien mit klar getrennten Verantwortlichkeiten:

```
modell.py      Song, Bibliothek
               Kein mutagen, kein tkinter, kein print()
               Reine Datenlogik -- direkt testbar

scanner.py     lese_song(), scanne_verzeichnis()
               Einzige Stelle die mutagen kennt
               Gibt Song-Objekte zurueck -- kein tkinter, kein print()

gui.py         MP3App, ScanDialog, StatistikFenster
               Importiert modell und scanner
               Kein mutagen direkt -- nur Song-Objekte
```

**Warum diese Trennung?** Weil `modell.py` so vollstaendig ohne GUI getestet werden kann. `pytest test_mp3.py` laeuft ohne Display, ohne Fenster, ohne mutagen-Dateien im Filesystem. Die Tests in `test_mp3.py` importieren nur `modell` -- und koennen alle Logik direkt pruefen.

In [ ]:
# Illustration der Abhaengigkeiten
# (kein ausfuehrbarer Code -- nur Darstellung)

ABHAENGIGKEITEN = """
test_mp3.py
    import modell       -> Song, Bibliothek testen
    import scanner      -> scanne_verzeichnis testen (mit echten Testdateien)

gui.py
    import modell       -> Bibliothek als Datenzustand
    import scanner      -> Scan in Thread starten
    import tkinter      -> Fenster, Canvas, Treeview

scanner.py
    import modell       -> Song-Objekte erstellen
    import mutagen      -> MP3-Dateien lesen

modell.py
    import json         -> Persistenz
    import pathlib      -> Pfad-Operationen
    import dataclasses  -> @dataclass, asdict
    -- keine weiteren Abhaengigkeiten --
"""
print(ABHAENGIGKEITEN)

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
11. Die GUI: Treeview als Songliste
</span>
</div>

Die zentrale GUI-Frage: Welches Widget zeigt eine Liste von Songs am besten?

`tk.Listbox` zeigt eine Liste von Strings -- keine Spalten. Fuer eine Songliste mit Titel, Interpret und Album brauchen wir ein tabellarisches Widget: `ttk.Treeview`.

`ttk.Treeview` ist tkinters eingebautes Tabellen-Widget. Es hat benannte Spalten, klickbare Header und eine Scrollbar.

In [ ]:
# Treeview -- Grundprinzip (nicht ausfuehren: braucht Display)
# Dieser Block zeigt die Struktur, kein laufendes Fenster.

TREEVIEW_KONZEPT = """
# Spalten definieren
tree = ttk.Treeview(
    root,
    columns=['titel', 'interpret', 'album', 'jahr', 'dauer'],
    show='headings'   # keine Icon-Spalte links
)

# Spaltenbreiten und Ausrichtung
tree.column('titel',     width=280, anchor='w')
tree.column('interpret', width=160, anchor='w')
tree.column('dauer',     width=55,  anchor='center')

# Klickbarer Header -- sortiert bei Klick
tree.heading('titel',     text='Titel',     command=lambda: sortiere_nach('titel'))
tree.heading('interpret', text='Interpret', command=lambda: sortiere_nach('interpret'))

# Zeilen einfuegen
for song in bibliothek.sortieren():
    tree.insert('', 'end', values=(
        song.titel_anzeige,
        song.interpret_anzeige,
        song.album_anzeige,
        song.jahr_anzeige,
        song.dauer_anzeige,
    ))

# Treeview leeren und neu befuellen (bei Suche oder Sortieraenderung)
tree.delete(*tree.get_children())   # alle Zeilen loeschen
# ...dann wieder einfuegen
"""
print(TREEVIEW_KONZEPT)

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
12. Das Thread-Problem beim Scannen
</span>
</div>

Wenn `scanne_verzeichnis()` direkt im Haupt-Thread aufgerufen wird, friert das Fenster ein -- tkinter kann keine Ereignisse verarbeiten solange Python-Code laeuft.

Die Loesung: den Scan in einem eigenen Thread starten. Das Ergebnis wird per `root.after()` in den Haupt-Thread zurueckgegeben -- das ist die einzige thread-sichere Methode um tkinter zu aktualisieren.

In [ ]:
# Das Thread-Muster fuer den Scan (vereinfacht, ohne laufendes Fenster)

THREAD_KONZEPT = """
import threading

def scan_starten():
    verzeichnis = filedialog.askdirectory()
    if not verzeichnis:
        return

    dialog = ScanDialog(root)   # Fortschrittsfenster oeffnen

    def im_thread():
        # Laeuft NICHT im Haupt-Thread.
        # Kein tkinter-Zugriff erlaubt!

        def fortschritt(aktuell, gesamt, name):
            # after() delegiert den Update in den Haupt-Thread
            root.after(0, dialog.aktualisieren, aktuell, gesamt, name)

        songs, fehler = scanne_verzeichnis(verzeichnis, fortschritt)

        # Ergebnis zurueck in den Haupt-Thread
        root.after(0, scan_fertig, songs, fehler, dialog)

    threading.Thread(target=im_thread, daemon=True).start()


def scan_fertig(songs, fehler, dialog):
    # Laeuft wieder im Haupt-Thread -- tkinter-Zugriff OK
    dialog.destroy()
    bibliothek.alle_hinzufuegen(songs)
    liste_aktualisieren()
"""

# Warum daemon=True?
# Ein Daemon-Thread wird automatisch beendet wenn das Hauptprogramm endet.
# Ohne daemon=True wuerde Python warten bis der Thread fertig ist --
# das Fenster koennte nicht geschlossen werden solange ein Scan laeuft.

print('Thread-Konzept (ohne laufendes Fenster -- nur zur Illustration)')
print(THREAD_KONZEPT)

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
13. Das Problem der veralteten Bibliothek
</span>
</div>

Die Bibliothek wird als JSON gespeichert. Was passiert wenn der Nutzer danach Dateien loescht oder verschiebt? Beim naechsten Start zeigt die Bibliothek Eintraege fuer Dateien die nicht mehr existieren.

Die Loesung ist einfach: beim Laden pruefen ob die gespeicherten Pfade noch existieren.

In [ ]:
# existiert()-Methode im Song: nutzt pathlib

import tempfile
from pathlib import Path

# Echte Datei anlegen, dann loeschen
tmp = Path(tempfile.mktemp(suffix='.mp3'))
tmp.write_bytes(b'dummy')

song_existiert = Song(str(tmp), 'Test', 'Interpret')
song_fehlt     = Song('/gibt/es/nicht.mp3', 'Geloescht', 'Jemand')

print(f'{song_existiert.titel}: existiert = {Path(song_existiert.pfad).is_file()}')
print(f'{song_fehlt.titel}:     existiert = {Path(song_fehlt.pfad).is_file()}')

# Bibliothek bereinigen: nur existierende Songs behalten
alle = [song_existiert, song_fehlt]
fehlende  = [s for s in alle if not Path(s.pfad).is_file()]
vorhandene = [s for s in alle if Path(s.pfad).is_file()]

print(f'\nBereinigt: {len(vorhandene)} vorhanden, {len(fehlende)} entfernt')
for s in fehlende:
    print(f'  Entfernt: {s.titel}')

tmp.unlink()   # aufraumen

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#80cbc4;background:rgba(128,203,196,0.1);border:1px solid rgba(128,203,196,0.25);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
14. Rueckblick: der Weg vom Problem zum Programm
</span>
</div>

| Schritt | Was wir getan haben |
|---|---|
| **1** | Das Problem konkret beschrieben: chaotische MP3-Sammlung, fehlende Metadaten |
| **2** | ID3-Tags verstanden: Schluessel-Wert-Paare, optional, verschiedene Formate |
| **3** | Naiven Tag-Zugriff ausprobiert -- drei Fehlerquellen identifiziert |
| **4** | Robuste Hilfsfunktionen gebaut: `None` statt Exception, `[:4]` fuer das Jahr |
| **5** | `@dataclass` als Datencontainer gewaehlt -- `__init__`, `__eq__` kostenlos, Properties fuer Fallbacks |
| **6** | JSON-Persistenz mit `asdict()` und gefilterten `aus_dict()`-Konstruktoren |
| **7** | Suche, Sortierung und Duplikatserkennung als Bibliothek-Klasse |
| **8** | `pathlib.rglob()` fuer rekursiven Scan, `dict.fromkeys()` fuer Deduplizierung |
| **9** | Fehlertoleranz: Fehler pro Datei sammeln statt den Scan abzubrechen |
| **10** | Drei Dateien mit sauberer Trennung: `modell`, `scanner`, `gui` |
| **11** | `ttk.Treeview` als Tabellen-Widget fuer die Songliste |
| **12** | Scan in eigenem Thread -- `root.after()` fuer thread-sicheres GUI-Update |
| **13** | Veraltete Eintraege: `Path.is_file()` prueft ob Dateien noch existieren |

Das zentrale Thema dieses Projekts war **Robustheit gegenueber realen Daten**. Eine Musiksammlung ist kein sauberer Datensatz -- Tags fehlen, Formate variieren, Dateien verschwinden. Jede dieser Situationen musste bedacht und abgefangen werden.

---

**Dateien des Projekts:**

```
modell.py        Song (@dataclass), Bibliothek -- kein I/O, kein GUI
scanner.py       lese_song(), scanne_verzeichnis() -- mutagen gekapselt
gui.py           tkinter-GUI mit Treeview, ScanDialog, Statistiken
test_mp3.py      pytest-Tests fuer Modell und Scanner
```

Starten: `python gui.py`